# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We will:

- Load Croissant metadata
- Review available RecordSets and Fields (referencing by their `@id`)
- Extract data for analysis and visualization
- Apply exploratory data analysis (EDA)

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install the mlcroissant library (uncomment if needed)
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and familiarize yourself with its schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata object
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Short description:", metadata.description)

# Print publishing information
print(f"Published: {getattr(metadata, 'datePublished', None)} | Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")

## 2. Data Overview
Let's examine available record sets and their fields. We always reference them by their `@id`.

In [ ]:
# Get RecordSets
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets, 1):
    print(f"Record set #{i} (@id): {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    print(f"  Description: {getattr(record_set, 'description', None)}")
    # Collect and print all available fields
    print(f"  Fields and Columns [referenced by @id]:")
    for field in record_set.fields:
        col_ids = [col.id for col in getattr(field, 'columns', [])]
        print(f"    - Field @id: {field.id} (name: {field.name}, columns: {col_ids})")
    print()

> **Note**: If nothing is displayed in the previous cell, check your schema's `record_sets`. The dataset may have no top-level record sets. In this case, consult the schema with `dataset.metadata.to_json()` and identify data file objects and their record sets by their `@id`.

For this dataset, let's enumerate **all record sets** referenced, so we can list all `@id` and use them for extraction.

In [ ]:
# List all RecordSet @ids
rs_ids = [rs.id for rs in dataset.metadata.record_sets]
print("RecordSet @ids:")
for rid in rs_ids:
    print(" -", rid)

## 3. Data Extraction
Let's load tabular records from specific RecordSets, referencing each by its `@id`.

We'll load all available RecordSets into pandas DataFrames for further analysis.

In [ ]:
# Prepare holders for DataFrames
dataframes = {}

# Use the IDs as an explicit list
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

if not record_set_ids:
    print("No record sets found in this schema. Please check the dataset schema for available record-oriented objects.")
else:
    for record_set_id in record_set_ids:
        print(f"Loading records for RecordSet: {record_set_id}")
        # records() returns a generator of dict records
        df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
        print(f"  Columns: {df.columns.tolist()}")
        dataframes[record_set_id] = df

    # Display the first 5 rows of the first (main) record set
    main_record_set_id = record_set_ids[0]
    print("\nSample data from RecordSet @id:", main_record_set_id)
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field and demonstrate filtering, normalization, and grouping. All operations reference fields/columns by their Croissant `@id`.

In [ ]:
# Pick the main record set and enumerate columns by @id for reference
main_df = dataframes[main_record_set_id]

# List columns to pick a numeric one (e.g., find IDs like 'peptide_count', 'coverage', etc.)
print("Available columns in main record set DataFrame:")
print(main_df.columns.tolist())

# Example: Let's pick a numeric column, e.g., 'cr:peptide_count' (replace by the real column @id shown above)
# For demonstration we'll guess a common one (update as necessary):
if 'cr:peptide_count' in main_df.columns:
    numeric_field_id = 'cr:peptide_count'
elif 'cr:coverage' in main_df.columns:
    numeric_field_id = 'cr:coverage'
else:
    # Just pick first numeric column
    numeric_field_id = main_df.select_dtypes('number').columns[0]

threshold = main_df[numeric_field_id].mean()  # Use mean as threshold in absence of prior info
filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):\n")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a categorical field (e.g., experiment condition)
possible_group_fields = [c for c in main_df.columns if main_df[c].dtype == object and len(main_df[c].unique()) < 10]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable group field found for grouping (categorical field with <10 unique values)")

## 5. Visualization
Let's visualize the distribution of the selected numeric field and relationships by group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field distribution (histogram)
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field_id], bins=30, kde=True, color='teal')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Grouped boxplot if grouping field exists
if 'group_field' in locals():
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
- We loaded and explored the FAIR^2 dataset using `mlcroissant`.
- We referenced all entities by their Croissant `@id` fields to ensure consistency and reproducibility.
- Initial EDA showed the distribution and spread of a main numeric measurement, and allowed for basic filtering and normalization.
- You can extend this workflow by exploring relationships between other columns, more sophisticated EDA, or building machine learning models based on the Croissant schema!